# Assignment

## Instructions

1.  **Regularization**:

    - Use the `diabetes` dataset from `sklearn.datasets`.
    - Compare the performance (Mean Squared Error) of `LinearRegression`, `Ridge`, and `Lasso` models.
    - Tune the `alpha` parameter for `Ridge` and `Lasso` using `GridSearchCV` with cross-validation to find the optimal regularization strength.

    ```python
    from sklearn.datasets import load_diabetes

    # Load the diabetes dataset
    diabetes = load_diabetes()
    ```

2.  **Ensemble Methods**:

    - Use the `breast_cancer` dataset from `sklearn.datasets`.
    - Compare the performance (F1 Score and AUC) of `DecisionTreeClassifier`, `RandomForestClassifier`, and `GradientBoostingClassifier`.
    - Tune the hyperparameters of each classifier using `GridSearchCV` with cross-validation.

    ```python
    from sklearn.datasets import load_breast_cancer

    # Load the breast cancer dataset
    breast_cancer = load_breast_cancer()
    ```

## Submission

- Submit the URL of the GitHub Repository that contains your work to NTU black board.
- Should you reference the work of your classmate(s) or online resources, give them credit by adding either the name of your classmate or URL.

In [20]:
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score, classification_report
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [21]:
# Q1
from sklearn.datasets import load_diabetes

# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# LinearRegression
linear = LinearRegression()
linear.fit(X_train, y_train)
linear_mse = mean_squared_error(y_test, linear.predict(X_test))

# Ridge
ridge_grid = GridSearchCV(
    estimator=Ridge(),
    param_grid={'alpha': [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='neg_mean_squared_error'            
)
ridge_grid.fit(X_train, y_train)
ridge_mse = mean_squared_error(y_test, ridge_grid.predict(X_test))

# Lasso
lasso_grid = GridSearchCV(
    estimator=Lasso(),
    param_grid={'alpha': [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='neg_mean_squared_error'
)
lasso_grid.fit(X_train, y_train)
lasso_mse = mean_squared_error(y_test, lasso_grid.predict(X_test))

print(f"Linear MSE: {linear_mse:.2f}")
print(f"Ridge  MSE: {ridge_mse:.2f}, best alpha={ridge_grid.best_params_}")
print(f"Lasso  MSE: {lasso_mse:.2f}, best alpha={lasso_grid.best_params_}")

Linear MSE: 3424.26
Ridge  MSE: 3435.80, best alpha={'alpha': 0.01}
Lasso  MSE: 3445.81, best alpha={'alpha': 0.01}


Conclusion

On the diabetes dataset, plain Linear Regression achieved the lowest test MSE (3424.26), outperforming both Ridge (3435.80, best alpha=0.01) and Lasso (3445.81, best alpha=0.01). Regularization did not improve generalization performance here, likely because the dataset has a favorable samples-to-features ratio (442 samples vs. 10 features), leaving little risk of overfitting for regularization to guard against. In this low-overfitting-risk setting, adding an L1 or L2 penalty only biases the coefficients away from the unconstrained least-squares solution without a corresponding benefit, resulting in a slightly higher MSE than unregularized Linear Regression.

In [25]:
# Q2
from sklearn.datasets import load_breast_cancer

breast_cancer = load_breast_cancer()
X, y = breast_cancer.data, breast_cancer.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


def evaluate(name, grid_search, X_test, y_test):
    grid_search.fit(X_train, y_train)
    preds = grid_search.predict(X_test)
    proba = grid_search.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, preds)                  
    auc = roc_auc_score(y_test, proba)             

    print(f"=== {name} ===")
    print(f"Best Params: {grid_search.best_params_}\n")
    print(classification_report(y_test, preds, target_names=breast_cancer.target_names)) 
    print(f"AUC: {auc:.4f}\n")

    return f1, auc

# ---------- Decision Tree ----------
dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(criterion='gini', random_state=0),
    param_grid={
        'max_depth': [None, 2, 4, 6, 8, 10],
        'min_samples_split': [2, 4, 6, 8],
        'min_samples_leaf': [2, 4, 6, 8],
    },
    cv=5, scoring='f1', n_jobs=-1
)

# ---------- Random Forest ----------
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=0),
    param_grid={
        'n_estimators': [100, 200, 300, 400, 500],
        'max_depth': [None, 2, 4, 6, 8],
        'min_samples_split': [2, 4, 6, 8],
        'min_samples_leaf': [2, 4, 6, 8],
    },
    cv=5, scoring='f1', n_jobs=-1
)

# ---------- Gradient Boosting ----------
gb_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=0),
    param_grid={
        'n_estimators': [100, 200, 300, 400, 500],
        'learning_rate': [0.01, 0.1, 0.2, 0.3],
        'subsample': [0.8, 0.9, 1.0],
        'min_samples_split': [2, 4, 6],
        'min_samples_leaf': [2, 4, 6],
    },
    cv=5, scoring='f1', n_jobs=-1
)

results = {}
results['Decision Tree']     = evaluate("Decision Tree",     dt_grid, X_test, y_test)
results['Random Forest']     = evaluate("Random Forest",     rf_grid, X_test, y_test)
results['Gradient Boosting'] = evaluate("Gradient Boosting", gb_grid, X_test, y_test)

print(f"{'Model':<20}{'F1':>10}{'AUC':>10}")
for name, (f1, auc) in results.items():
    print(f"{name:<20}{f1:>10.4f}{auc:>10.4f}")

=== Decision Tree ===
Best Params: {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2}

              precision    recall  f1-score   support

   malignant       0.90      0.96      0.93        47
      benign       0.97      0.93      0.95        67

    accuracy                           0.94       114
   macro avg       0.93      0.94      0.94       114
weighted avg       0.94      0.94      0.94       114

AUC: 0.9935

=== Random Forest ===
Best Params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

              precision    recall  f1-score   support

   malignant       0.96      0.96      0.96        47
      benign       0.97      0.97      0.97        67

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114

AUC: 0.9965

=== Gradient Boosting ===
Best Params: {'learning_rate': 0.1, 'min_samples_leaf': 6, 'min_sam

Conclusion

On the breast cancer dataset, all three ensemble models performed well, with a clear ranking on both F1 and AUC: Gradient Boosting (F1=0.978, AUC=0.998) > Random Forest (F1=0.970, AUC=0.997) > Decision Tree (F1=0.947, AUC=0.994).

This matches the expected pattern for the two ensemble strategies: Random Forest's improvement over the single Decision Tree comes from variance reduction (bagging — averaging many independently trained trees smooths out the instability of any single tree), while Gradient Boosting's further improvement over Random Forest comes from bias reduction (boosting — each tree sequentially corrects the errors of the previous ensemble).

Looking at the per-class breakdown, all three models achieve a similar recall (~0.96) on the malignant class, meaning they catch true malignant cases at roughly the same rate. The main difference is precision on malignant: the single Decision Tree has more false positives (precision=0.90) than Random Forest (0.96) and Gradient Boosting (0.98), so the ensembles mainly reduce false alarms rather than missed diagnoses.